# Modelo v2 – Regresión logística (modelos A y B)

**Proyecto:** Cumplimiento crediticio de los hogares colombianos (ELCO 2022) · **Versión:** v2 (protocolo común del grupo)

## Configuración

In [ ]:
import os, json, warnings
from hashlib import sha256
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import mlflow

import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
SK = tuple(int(x) for x in sklearn.__version__.split(".")[:2])

SEED = 42
N_FOLDS = 4
TARGET, GROUP = "objetivo_atraso", "DIRECTORIO"

EN_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ
if EN_DATABRICKS:
    DATA_PATH = "<ruta del CSV en Databricks>"
    EXPERIMENT = "/elco-estado-de-pago-RaulM"
    OUT = Path("/tmp/resultados_logistica_E3")
else:
    DATA_PATH = r"C:\Users\Raul M\Desktop\MAESTRIA\CICLOS\CICLO 6\DESPLIEGUE_SOLUCIONES\Proyecto\DATOS\ELCO_2022_creditos_ahorro_laboral_consolidado.csv"
    EXPERIMENT = "elco-estado-de-pago-V2-RaulM"
    mlflow.set_tracking_uri("http://98.93.104.85:8050") 
    OUT = Path("resultados_logistica_E3")
OUT.mkdir(parents=True, exist_ok=True)
mlflow.set_experiment(EXPERIMENT)

HASH_DATOS = sha256(Path(DATA_PATH).read_bytes()).hexdigest()
print("MLflow:", mlflow.get_tracking_uri(), "| experimento:", EXPERIMENT, "| sklearn", sklearn.__version__)

## Carga y variables derivadas

In [ ]:
raw = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df = raw.copy()
assert df[GROUP].nunique() == 3020 and (df[TARGET] == (df["P2110"] == 2).astype(int)).all()

CATEGORICAS = ["P2097", "P2098", "P2106"]
NUMERICAS_A = ["P2099S2", "P2104", "P2105S1", "P2107S1"]
DINERO = ["P2104", "P2105S1"]
HOGAR = ["n_personas_ahorro", "n_personas_con_ingresos", "n_personas_que_ahorran",
         "prop_que_ahorran_sobre_con_ingresos", "n_personas_con_cuenta",
         "n_personas_laboral", "n_personas_trabajo_semana"]
VARIABLES = {"A": CATEGORICAS + NUMERICAS_A, "B": CATEGORICAS + NUMERICAS_A + HOGAR}
assert not {"P2110", "P2108", GROUP, TARGET} & set(VARIABLES["B"])

df["P2107S1"] = pd.to_numeric(df["P2107S1"], errors="coerce").replace(9999, np.nan)
for c in NUMERICAS_A + HOGAR:
    df[c] = pd.to_numeric(df[c], errors="coerce").astype("float64")
df[TARGET] = df[TARGET].astype(int)

print(f"Créditos: {len(df):,} | hogares: {df[GROUP].nunique():,} | tasa base de retraso: {df[TARGET].mean():.4f}")
print({k: len(v) for k, v in VARIABLES.items()}, "variables (A, B)")

## Partición por hogar: entrenamiento 70 % / validación 15 % / prueba 15 %

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=.30, random_state=SEED)
i_train, i_temp = next(gss.split(df, df[TARGET], groups=df[GROUP]))
train, temporal = df.iloc[i_train].copy(), df.iloc[i_temp].copy()
gss2 = GroupShuffleSplit(n_splits=1, test_size=.50, random_state=SEED + 1)
i_val, i_test = next(gss2.split(temporal, temporal[TARGET], groups=temporal[GROUP]))
val, test = temporal.iloc[i_val].copy(), temporal.iloc[i_test].copy()
for p1, p2 in [(train, val), (train, test), (val, test)]:
    assert set(p1[GROUP]).isdisjoint(set(p2[GROUP]))
for nombre, parte in [("Entrenamiento", train), ("Validación", val), ("Prueba", test)]:
    assert parte[TARGET].nunique() == 2, f"Falta una clase en {nombre}"
    print(f"{nombre}: {len(parte)} créditos, {parte[GROUP].nunique()} hogares, retrasos={parte[TARGET].mean():.3%}")

particiones = pd.concat([
    pd.DataFrame({"indice_fila": p.index, "DIRECTORIO": p[GROUP].to_numpy(),
                  "particion": nombre, "objetivo_atraso": p[TARGET].to_numpy()})
    for nombre, p in [("train", train), ("val", val), ("test", test)]
]).sort_values("indice_fila")
particiones.to_csv(OUT / "particiones_comunes_grupo19.csv", index=False)
y_train, y_val, y_test = (p[TARGET].astype(int) for p in (train, val, test))

## Pipeline y metricas

In [ ]:
def make_lr(reg, C, class_weight):
    base = dict(C=C, class_weight=class_weight, max_iter=5000, random_state=SEED)
    if reg == "l2":
        return LogisticRegression(**base)                       # lbfgs, L2 (por defecto)
    l1 = 1.0 if reg == "l1" else 0.5                            # elastic net: mitad L1, mitad L2
    if SK >= (1, 8):                                            # API nueva: l1_ratio reemplaza a 'penalty'
        return LogisticRegression(solver="saga", l1_ratio=l1, **base)
    return LogisticRegression(solver="saga", penalty="l1" if reg == "l1" else "elasticnet",
                              **({} if reg == "l1" else {"l1_ratio": l1}), **base)

def build_pipeline(fs, hp):
    variables = VARIABLES[fs]
    cats = [c for c in CATEGORICAS if c in variables]
    dinero = [c for c in DINERO if c in variables]
    nums = [c for c in variables if c not in cats and c not in dinero]
    def numeric(log=False):
        pasos = [("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one"))] if log else []
        pasos += [("impute", SimpleImputer(strategy="median", add_indicator=True)), ("scale", StandardScaler())]
        return Pipeline(pasos)
    prep = ColumnTransformer([
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=hp["min_frequency"],
                                                sparse_output=False))]), cats),
        ("money", numeric(log=True), dinero),
        ("num", numeric(), nums)], remainder="drop", verbose_feature_names_out=False)
    return Pipeline([("preparacion", prep), ("modelo", make_lr(hp["reg"], hp["C"], hp["class_weight"]))])

UMBRALES = np.round(np.arange(0.10, 0.901, 0.025), 3)

def seleccionar_umbral(y_real, prob):
    scores = np.array([f1_score(y_real, prob >= u, zero_division=0) for u in UMBRALES])
    return float(UMBRALES[int(np.argmax(scores))])

def calcular_metricas(y_real, prob, umbral):
    pred = (prob >= umbral).astype(int)
    return {"pr_auc": float(average_precision_score(y_real, prob)), "roc_auc": float(roc_auc_score(y_real, prob)),
            "precision": float(precision_score(y_real, pred, zero_division=0)),
            "recall": float(recall_score(y_real, pred, zero_division=0)),
            "f1": float(f1_score(y_real, pred, zero_division=0)), "umbral": float(umbral)}

def priorizacion(y_real, prob):
    """Si el analista revisa el 10/20/30 % de créditos con mayor probabilidad: cuántos retrasos encuentra."""
    filas, y_real = [], np.asarray(y_real)
    orden = np.argsort(-prob, kind="stable")
    y_ord = y_real[orden]
    for fraccion in [0.10, 0.20, 0.30]:
        n = max(1, int(np.ceil(len(y_real) * fraccion)))
        enc = int(y_ord[:n].sum())
        filas.append({"porcentaje_revisado": int(fraccion * 100), "creditos_revisados": n, "retrasos_encontrados": enc,
                      "precision_top": enc / n, "recall_top": enc / int(y_real.sum()),
                      "creditos_por_retraso": n / enc if enc else np.nan})
    return pd.DataFrame(filas)

## Experimentos

In [ ]:
HP = {"reg": "elasticnet",
      "C": 0.03,
      "class_weight": "balanced",
      "min_frequency": 20}

HP_TAG = f"{HP['reg']}_C-{HP['C']}_cw-{HP['class_weight']}_mf-{HP['min_frequency']}"
cv = GroupKFold(n_splits=N_FOLDS)
filas_exp = []
for fs in ("A", "B"):
    cols = VARIABLES[fs]
    pipe = build_pipeline(fs, HP).fit(train[cols], y_train)
    proba = pipe.predict_proba(val[cols])[:, 1]
    umbral = seleccionar_umbral(y_val, proba)                       # umbral: solo con validación
    met = calcular_metricas(y_val, proba, umbral)
    cvs = cross_val_score(build_pipeline(fs, HP), train[cols], y_train, groups=train[GROUP], cv=cv,
                          scoring="average_precision")              # CV agrupada por hogar, solo entrenamiento
    with mlflow.start_run(run_name=f"EXP_{fs}_{HP_TAG}"):
        mlflow.set_tags({"integrante": "Raul", "modelo": "LogisticRegression", "etapa": "experimento",
                         "hash_datos": HASH_DATOS[:16], "split_seed": str(SEED)})
        mlflow.log_params({"grupo": fs, **{k: str(v) for k, v in HP.items()}})
        mlflow.log_metrics({**{f"val_{k}": v for k, v in met.items()},
                            "cv_pr_auc": float(cvs.mean()), "cv_std_pr_auc": float(cvs.std())})
    filas_exp.append({"grupo": fs, **met, "cv_pr_auc": cvs.mean()})
pd.DataFrame(filas_exp).round(4)

## Tabla de experimentos en validación

In [ ]:
exp = mlflow.get_experiment_by_name(EXPERIMENT)
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id],
                          filter_string="tags.modelo = 'LogisticRegression' and tags.etapa = 'experimento'")
cols = {"params.grupo": "grupo", "params.reg": "reg", "params.C": "C", "params.class_weight": "class_weight",
        "params.min_frequency": "min_frequency", "metrics.val_pr_auc": "pr_auc", "metrics.val_roc_auc": "roc_auc",
        "metrics.val_precision": "precision", "metrics.val_recall": "recall", "metrics.val_f1": "f1",
        "metrics.val_umbral": "umbral", "metrics.cv_pr_auc": "cv_pr_auc", "metrics.cv_std_pr_auc": "cv_std_pr_auc"}
tabla = runs[list(cols)].rename(columns=cols)
for c in tabla.columns[5:]:
    tabla[c] = tabla[c].astype(float).round(4)
tabla = tabla.sort_values(["pr_auc", "f1"], ascending=[False, False]).reset_index(drop=True)
tabla.to_csv(OUT / "experimentos_validacion_logistica.csv", index=False)
tabla

## Comparación A contra B por configuración

In [ ]:
par = tabla.pivot_table(index=["reg", "C", "class_weight", "min_frequency"], columns="grupo",
                        values=["pr_auc", "recall", "precision", "f1"])
comp = pd.DataFrame({
    "d_pr_auc":   par["pr_auc"]["B"] - par["pr_auc"]["A"],
    "d_recall":   par["recall"]["B"] - par["recall"]["A"],
    "d_precision": par["precision"]["B"] - par["precision"]["A"],
    "d_f1":       par["f1"]["B"] - par["f1"]["A"]}).round(4)
comp["cumple_criterio"] = (comp["d_recall"] >= 0.05) & (comp["d_precision"] >= -0.03)
comp.to_csv(OUT / "comparacion_A_vs_B_validacion.csv")
comp

## Comparación A contra B: mejor modelo de cada grupo

In [ ]:
claves = ["reg", "C", "class_weight", "min_frequency"]
TA = tabla[tabla["grupo"] == "A"].set_index(claves)
TB = tabla[tabla["grupo"] == "B"].set_index(claves)
mejorA = tabla[tabla["grupo"] == "A"].iloc[0]
mejorB = tabla[tabla["grupo"] == "B"].iloc[0]

def cfg(s):
    return f"{s['reg']}, C={s['C']}, cw={s['class_weight']}, mf={s['min_frequency']}"

def fila(caso, cfg_a, cfg_b, a, b):
    d_rec, d_pre = b["recall"] - a["recall"], b["precision"] - a["precision"]
    return {"caso": caso, "config_A": cfg_a, "config_B": cfg_b,
            "pr_auc_A": a["pr_auc"], "pr_auc_B": b["pr_auc"],
            "recall_A": a["recall"], "recall_B": b["recall"],
            "precision_A": a["precision"], "precision_B": b["precision"],
            "f1_A": a["f1"], "f1_B": b["f1"],
            "umbral_A": a["umbral"], "umbral_B": b["umbral"],
            "d_pr_auc": b["pr_auc"] - a["pr_auc"], "d_recall": d_rec, "d_precision": d_pre,
            "d_f1": b["f1"] - a["f1"], "cumple_criterio": bool(d_rec >= 0.05 and d_pre >= -0.03)}

b_de_mejorA = TB.loc[tuple(mejorA[claves])]
a_de_mejorB = TA.loc[tuple(mejorB[claves])]

pares = pd.DataFrame([
    fila("Mejor A y su B (misma config.)", cfg(mejorA), cfg(mejorA), mejorA, b_de_mejorA),
    fila("Mejor B y su A (misma config.)", cfg(mejorB), cfg(mejorB), a_de_mejorB, mejorB),
    fila("Mejor A contra mejor B", cfg(mejorA), cfg(mejorB), mejorA, mejorB),
]).round(4)
pares.to_csv(OUT / "pares_A_vs_B_validacion.csv", index=False)
print("Guardado en:", OUT / "pares_A_vs_B_validacion.csv")
pares.T

## Figura: PR-AUC de validación por configuración

In [ ]:
from matplotlib.patches import Patch

def figura_experimentos(xlim, archivo):
    t = tabla.sort_values("pr_auc", ascending=True).reset_index(drop=True)
    colores = ["#2a6f97" if g == "A" else "#e07a1f" for g in t["grupo"]]
    etiquetas = [f"{r.grupo} · {r.reg} | C={r.C} | cw={r.class_weight} | mf={r.min_frequency}" for r in t.itertuples()]
    azar = float(y_val.mean())
    fig, ax = plt.subplots(figsize=(9, 8))
    ax.barh(range(len(t)), t["pr_auc"], color=colores)
    for i, v in enumerate(t["pr_auc"]):
        ax.text(v + (xlim[1] - xlim[0]) * 0.004, i, f"{v:.4f}", va="center", fontsize=8)
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(etiquetas, fontsize=8)
    if xlim[0] <= azar <= xlim[1]:
        ax.axvline(azar, color="gray", ls="--", lw=1)
        ax.text(azar + (xlim[1] - xlim[0]) * 0.004, len(t) - 0.4, f"azar = {azar:.3f}", color="gray", fontsize=8)
    ax.set_xlim(*xlim)
    ax.set_xlabel(f"PR-AUC en validación (el eje empieza en {xlim[0]})")
    ax.set_title("Regresión logística: PR-AUC de validación por configuración")
    ax.legend(handles=[Patch(color="#2a6f97", label="A: variables del crédito"),
                       Patch(color="#e07a1f", label="B: crédito + hogar")], loc="lower right")
    plt.tight_layout()
    plt.savefig(OUT / archivo, dpi=200)
    plt.show()

figura_experimentos((0.14, 0.32), "comparacion_experimentos_val_pr_auc_completa.png")   # incluye la línea de azar
figura_experimentos((0.28, 0.31), "comparacion_experimentos_val_pr_auc_zoom.png")       # detalle de las diferencias

## Resumen del mejor modelo de cada grupo

In [ ]:
filas = []
for g in ("A", "B"):
    m = tabla[tabla["grupo"] == g].iloc[0]
    filas.append({"modelo": "LogisticRegression", "grupo": g,
                  "configuracion": f"{m['reg']}, C={m['C']}, cw={m['class_weight']}, mf={m['min_frequency']}",
                  "pr_auc": m["pr_auc"], "roc_auc": m["roc_auc"], "precision": m["precision"],
                  "recall": m["recall"], "f1": m["f1"], "umbral": m["umbral"], "cv_pr_auc": m["cv_pr_auc"]})
resumen = pd.DataFrame(filas)
resumen.to_csv(OUT / "resumen_mejor_validacion_logistica.csv", index=False)
resumen

## Modelo final y evaluación en prueba

Se evalúa en prueba el mejor A y el mejor B, cada uno elegido solo con validación, y se calcula el criterio de éxito (Δrecall ≥ +0.05 y Δprecisión ≥ −0.03). El modelo final del algoritmo es el de mayor PR-AUC en validación.

In [ ]:
# Se ejecuta UNA SOLA VEZ, cuando los experimentos ya terminaron y la tabla está lista.
resultados = {}
for gname in ("A", "B"):
    g = tabla[tabla["grupo"] == gname].iloc[0]          # mejor del grupo por PR-AUC de validación (desempate: F1)
    hp = {"reg": g["reg"], "C": float(g["C"]), "class_weight": None if g["class_weight"] == "None" else g["class_weight"],
          "min_frequency": int(g["min_frequency"])}
    cols_g = VARIABLES[gname]
    modelo = build_pipeline(gname, hp).fit(train[cols_g], y_train)
    umbral = seleccionar_umbral(y_val, modelo.predict_proba(val[cols_g])[:, 1])      # umbral: solo con validación
    prob = modelo.predict_proba(test[cols_g])[:, 1]
    resultados[gname] = {"nombre": f"LR_{gname}_{hp['reg']}_C-{hp['C']}_cw-{hp['class_weight']}_mf-{hp['min_frequency']}",
                         "hp": hp, "cols": cols_g, "modelo": modelo, "umbral": umbral, "prob": prob,
                         "val_pr_auc": float(g["pr_auc"]), "val_f1": float(g["f1"]),
                         "met": calcular_metricas(y_test, prob, umbral), "prior": priorizacion(y_test, prob)}

# ---- comparación A vs B en prueba y criterio de éxito ----
mA, mB = resultados["A"]["met"], resultados["B"]["met"]
d_rec, d_pre = mB["recall"] - mA["recall"], mB["precision"] - mA["precision"]
cumple = bool(d_rec >= 0.05 and d_pre >= -0.03)
comp_test = pd.DataFrame({"A": mA, "B": mB}).T.round(4)
print(comp_test.to_string())
print(f"\nΔrecall (B−A) = {d_rec:+.4f}   Δprecisión (B−A) = {d_pre:+.4f}   "
      f"ΔF1 = {mB['f1'] - mA['f1']:+.4f}   ΔPR-AUC = {mB['pr_auc'] - mA['pr_auc']:+.4f}")
print("CRITERIO DE ÉXITO (Δrecall ≥ +0.05 y Δprecisión ≥ −0.03):", "SÍ SE CUMPLE" if cumple else "NO SE CUMPLE")
for gname in ("A", "B"):
    print(f"\nPriorización {gname} (prueba):"); print(resultados[gname]["prior"].round(3).to_string(index=False))

# ---- modelo final del algoritmo: el de mayor PR-AUC en validación (desempate F1) ----
ganador = max(resultados, key=lambda k: (resultados[k]["val_pr_auc"], resultados[k]["val_f1"]))
r = resultados[ganador]
print("\nMODELO FINAL (por validación):", r["nombre"])

comp_test.assign(delta_recall=d_rec, delta_precision=d_pre, cumple_criterio=cumple).to_csv(OUT / "comparacion_A_vs_B_prueba.csv")
pd.DataFrame([{"nombre": r["nombre"], "grupo": ganador, **r["met"]}]).to_csv(OUT / "metricas_finales_prueba.csv", index=False)
r["prior"].to_csv(OUT / "priorizacion_creditos_prueba.csv", index=False)
joblib.dump(r["modelo"], OUT / "regresion_logistica_pipeline_final.joblib")
metadata = {"algoritmo": "LogisticRegression", "modelo_seleccionado": r["nombre"], "grupo": ganador,
            "variables": r["cols"], "categoricas": [c for c in CATEGORICAS if c in r["cols"]],
            "hiperparametros": r["hp"], "umbral": r["umbral"], "semilla": SEED, "sha256_csv": HASH_DATOS,
            "metrica_seleccion": "pr_auc_validacion", "metricas_prueba": r["met"], "tasa_base_prueba": float(y_test.mean()),
            "comparacion_AB": {"delta_recall": d_rec, "delta_precision": d_pre, "criterio_cumplido": cumple}}
(OUT / "metadata_modelo_final.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")

for gname in ("A", "B"):
    x = resultados[gname]
    with mlflow.start_run(run_name=f"FINAL_{x['nombre']}"):
        mlflow.set_tags({"integrante": "Raul", "modelo": "LogisticRegression",
                         "fase": "modelo_final" if gname == ganador else "comparacion_AB",
                         "hash_datos": HASH_DATOS[:16], "split_seed": str(SEED)})
        mlflow.log_params({"nombre": x["nombre"], "grupo": gname, "umbral": x["umbral"], "seed": SEED,
                           **{k: str(v) for k, v in x["hp"].items()}})
        mlflow.log_metrics({f"test_{k}": float(v) for k, v in x["met"].items()})
        if gname == ganador:
            for f in ["regresion_logistica_pipeline_final.joblib", "metadata_modelo_final.json",
                      "experimentos_validacion_logistica.csv", "pares_A_vs_B_validacion.csv",
                      "comparacion_A_vs_B_validacion.csv", "metricas_finales_prueba.csv",
                      "priorizacion_creditos_prueba.csv", "comparacion_A_vs_B_prueba.csv",
                      "particiones_comunes_grupo19.csv"]:
                mlflow.log_artifact(str(OUT / f))
assert np.allclose(joblib.load(OUT / "regresion_logistica_pipeline_final.joblib").predict_proba(test[r["cols"]])[:, 1], r["prob"])
print("LISTO. Archivos en:", OUT.resolve())